# Backpropagation

**Learning Objectives:** Understand the chain rule, backpropagation algorithm, computational graphs, gradient descent variants, weight initialization, and learning rate scheduling. Implement backprop from scratch.

## 1. Chain Rule for Gradients

Backpropagation applies the **chain rule** to compute gradients of the loss w.r.t. every parameter.

For a composition $f(g(x))$:
$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$

For a multi-variable chain:
$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$

where $z = wx + b$, $a = \sigma(z)$, $L = \text{loss}(a, y)$

## 2. Computational Graphs

A computational graph represents the forward computation as a directed acyclic graph (DAG).

**Forward pass:** compute outputs left-to-right
**Backward pass:** compute gradients right-to-left (reverse-mode autodiff)

Example for $f = (w*x + b) * 2$:

```
x → (*) → (+) → (*2) → f
w →   b →   
```

In [ ]:
# Manual gradient computation with a simple computational graph
import numpy as np

# Forward: f = (w * x + b) * 2
w, x, b = 3.0, 2.0, 1.0

z = w * x + b        # z = 3*2 + 1 = 7
f = z * 2            # f = 14

# Backward (reverse-mode):
df_dz = 2.0           # df/dz = 2
dz_dw = x             # dz/dw = x = 2
dz_db = 1.0           # dz/db = 1
df_dw = df_dz * dz_dw # df/dw = 2 * 2 = 4
df_db = df_dz * dz_db # df/db = 2 * 1 = 2

print(f"f = {f}")
print(f"df/dw = {df_dw}, df/db = {df_db}")

## 3. Backpropagation Algorithm Step-by-Step

For a 2-layer network: $X \to W_1, b_1 \to Z_1 \to A_1 \to W_2, b_2 \to Z_2 \to A_2 \to Loss$

**Forward pass:**
1. $Z_1 = W_1 X + b_1$
2. $A_1 = \text{ReLU}(Z_1)$
3. $Z_2 = W_2 A_1 + b_2$
4. $A_2 = \sigma(Z_2)$
5. $L = \text{BCE}(Y, A_2)$

**Backward pass:**
1. $dZ_2 = A_2 - Y$
2. $dW_2 = \frac{1}{m} dZ_2 A_1^T$
3. $db_2 = \frac{1}{m} \sum dZ_2$
4. $dA_1 = W_2^T dZ_2$
5. $dZ_1 = dA_1 \odot [Z_1 > 0]$
6. $dW_1 = \frac{1}{m} dZ_1 X^T$
7. $db_1 = \frac{1}{m} \sum dZ_1$

In [ ]:
# Full backpropagation implementation from scratch
np.random.seed(42)

def initialize_weights(layer_dims, init_method='random'):
    params = {}
    for l in range(1, len(layer_dims)):
        if init_method == 'xavier':
            limit = np.sqrt(6 / (layer_dims[l-1] + layer_dims[l]))
            params[f'W{l}'] = np.random.uniform(-limit, limit, (layer_dims[l], layer_dims[l-1]))
        elif init_method == 'he':
            params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2 / layer_dims[l-1])
        else:  # random small
            params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        params[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return params

def forward_prop(X, params):
    cache = {'A0': X}
    L = len(params) // 2
    for l in range(1, L + 1):
        cache[f'Z{l}'] = params[f'W{l}'] @ cache[f'A{l-1}'] + params[f'b{l}']
        if l < L:
            cache[f'A{l}'] = np.maximum(cache[f'Z{l}'], 0)  # ReLU
        else:
            cache[f'A{l}'] = 1 / (1 + np.exp(-cache[f'Z{l}']))  # Sigmoid
    return cache[f'A{L}'], cache

def back_prop(Y, cache, params):
    grads = {}
    L = len(params) // 2
    m = Y.shape[1]

    # Output layer gradient
    dZ = cache[f'A{L}'] - Y
    for l in range(L, 0, -1):
        grads[f'dW{l}'] = (1 / m) * dZ @ cache[f'A{l-1}'].T
        grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
        if l > 1:  # backprop to previous layer
            dA = params[f'W{l}'].T @ dZ
            dZ = dA * (cache[f'Z{l-1}'] > 0)  # ReLU derivative
    return grads

def update_params(params, grads, lr):
    L = len(params) // 2
    for l in range(1, L + 1):
        params[f'W{l}'] -= lr * grads[f'dW{l}']
        params[f'b{l}'] -= lr * grads[f'db{l}']
    return params

def train(X, Y, layer_dims, lr=0.1, epochs=1000, init='he'):
    params = initialize_weights(layer_dims, init)
    losses = []
    for i in range(epochs):
        A, cache = forward_prop(X, params)
        eps = 1e-15
        loss = -np.mean(Y * np.log(np.clip(A, eps, 1-eps)) + (1-Y) * np.log(np.clip(1-A, eps, 1-eps)))
        losses.append(loss)
        grads = back_prop(Y, cache, params)
        params = update_params(params, grads, lr)
    return params, losses

In [ ]:
# Test backprop on make_moons
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X = X.T
y = y.reshape(1, -1)

params, losses = train(X, y, layer_dims=[2, 16, 8, 1], lr=0.3, epochs=2000)

print(f"Final loss: {losses[-1]:.4f}")

import matplotlib.pyplot as plt
plt.plot(losses)
plt.title('Backprop Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.show()

## 4. Gradient Descent Variants

| Variant | Update Rule | Pros | Cons |
|---------|-------------|------|------|
| **Batch GD** | $\theta = \theta - \eta \nabla_\theta L(\theta)$ | Stable convergence | Slow on large datasets |
| **Stochastic GD** | $\theta = \theta - \eta \nabla_\theta L(\theta; x^{(i)}, y^{(i)})$ | Fast updates, escapes local minima | Noisy, oscillates |
| **Mini-batch GD** | $\theta = \theta - \eta \nabla_\theta L(\theta; X^{(b)}, Y^{(b)})$ | Balance of speed & stability | Need to tune batch size |

Typical batch sizes: 32, 64, 128, 256.

In [ ]:
# Mini-batch gradient descent implementation
def create_mini_batches(X, Y, batch_size=32):
    m = X.shape[1]
    indices = np.random.permutation(m)
    batches = []
    for i in range(0, m, batch_size):
        batch_idx = indices[i:min(i+batch_size, m)]
        batches.append((X[:, batch_idx], Y[:, batch_idx]))
    return batches

def train_minibatch(X, Y, layer_dims, lr=0.1, epochs=100, batch_size=32):
    params = initialize_weights(layer_dims, 'he')
    losses = []
    for i in range(epochs):
        batches = create_mini_batches(X, Y, batch_size)
        epoch_loss = 0
        for X_batch, Y_batch in batches:
            A, cache = forward_prop(X_batch, params)
            eps = 1e-15
            loss = -np.mean(Y_batch * np.log(np.clip(A, eps, 1-eps)) +
                          (1-Y_batch) * np.log(np.clip(1-A, eps, 1-eps)))
            epoch_loss += loss * X_batch.shape[1]
            grads = back_prop(Y_batch, cache, params)
            params = update_params(params, grads, lr)
        losses.append(epoch_loss / X.shape[1])
    return params, losses

## 5. Vanishing & Exploding Gradients

**Vanishing gradients:** Early layers get tiny gradients → slow or no learning.
- Cause: repeated multiplication of small gradients (sigmoid/tanh saturate)
- Fix: ReLU, batch norm, residual connections

**Exploding gradients:** Early layers get huge gradients → unstable training.
- Cause: repeated multiplication of large values
- Fix: gradient clipping, proper initialization, batch norm

### Weight Initialization

| Method | Distribution | Best For |
|--------|-------------|----------|
| **Xavier/Glorot** | $\mathcal{U}(-\sqrt{\frac{6}{n_{in}+n_{out}}}, \sqrt{\frac{6}{n_{in}+n_{out}}})$ | Sigmoid, Tanh |
| **He** | $\mathcal{N}(0, \sqrt{\frac{2}{n_{in}}})$ | ReLU, Leaky ReLU |
| **LeCun** | $\mathcal{N}(0, \sqrt{\frac{1}{n_{in}}})$ | SELU |

In [ ]:
# Compare initialization strategies
def compare_init(strategies, layer_dims, X, Y, epochs=500, lr=0.1):
    results = {}
    for method in strategies:
        params = initialize_weights(layer_dims, method)
        losses = []
        for _ in range(epochs):
            A, cache = forward_prop(X, params)
            eps = 1e-15
            loss = -np.mean(Y * np.log(np.clip(A, eps, 1-eps)) +
                          (1-Y) * np.log(np.clip(1-A, eps, 1-eps)))
            losses.append(loss)
            grads = back_prop(Y, cache, params)
            params = update_params(params, grads, lr)
        results[method] = losses
        print(f"{method:10s} final loss: {losses[-1]:.4f}")
    return results

# Deeper network to highlight differences
X_small, y_small = make_moons(n_samples=200, noise=0.15, random_state=42)
X_small, y_small = X_small.T, y_small.reshape(1, -1)

results = compare_init(['random', 'xavier', 'he'], [2, 32, 32, 16, 1], X_small, y_small, epochs=800)

plt.figure(figsize=(10, 5))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.title('Weight Initialization Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Learning Rate Scheduling

Adjusting the learning rate during training improves convergence:

**Step decay:** $\eta_t = \eta_0 \cdot \gamma^{\lfloor t / step \rfloor}$

**Exponential decay:** $\eta_t = \eta_0 \cdot e^{-kt}$

**Cosine decay:** $\eta_t = \eta_{min} + \frac{1}{2}(\eta_0 - \eta_{min})(1 + \cos(\frac{t}{T}\pi))$

**Reduce on Plateau:** Reduce LR when validation loss stops improving.

In [ ]:
# Implement learning rate schedules
def step_decay(epoch, initial_lr=0.1, drop=0.5, epochs_drop=100):
    return initial_lr * (drop ** np.floor(epoch / epochs_drop))

def exp_decay(epoch, initial_lr=0.1, k=0.01):
    return initial_lr * np.exp(-k * epoch)

def cosine_decay(epoch, initial_lr=0.1, total_epochs=500):
    return 0.0001 + 0.5 * (initial_lr - 0.0001) * (1 + np.cos(np.pi * epoch / total_epochs))

epochs = 300
lrs_step = [step_decay(e) for e in range(epochs)]
lrs_exp = [exp_decay(e) for e in range(epochs)]
lrs_cos = [cosine_decay(e, total_epochs=epochs) for e in range(epochs)]

plt.figure(figsize=(12, 4))
plt.plot(lrs_step, label='Step Decay')
plt.plot(lrs_exp, label='Exponential Decay')
plt.plot(lrs_cos, label='Cosine Decay')
plt.title('Learning Rate Schedules')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Backpropagation in Keras

Keras handles backprop automatically via **automatic differentiation** (tf.GradientTape).

In [ ]:
# Automatic differentiation with tf.GradientTape
import tensorflow as tf

# Define a simple computation
x = tf.Variable(3.0)
w = tf.Variable(2.0)
b = tf.Variable(1.0)

with tf.GradientTape() as tape:
    z = w * x + b
    loss = z ** 2  # L = (wx + b)^2

# Compute gradients automatically
grads = tape.gradient(loss, [w, b])
print(f"dL/dw = {grads[0].numpy():.4f}")
print(f"dL/db = {grads[1].numpy():.4f}")

# Verify manually: dL/dw = 2(wx+b)*x = 2*(2*3+1)*3 = 42
print(f"Expected dL/dw = {2 * (2*3 + 1) * 3:.4f}")

## 8. Interview Problems

### Problem 1: Implement Backprop for a 2-Layer Network Manually

Implement a 2-layer network (input→hidden→output) with ReLU hidden and sigmoid output. Compute forward pass, then compute all gradients manually using the chain rule. Verify with tf.GradientTape.

### Problem 2: Compare Initialization Strategies

Train networks with random, Xavier, and He initialization on a deep network (5+ layers). Show that poor initialization leads to vanishing/exploding gradients by plotting activation variances.

### Problem 3: Gradient Clipping

Implement gradient clipping by norm and by value during backprop to prevent exploding gradients.

In [ ]:
# Solution: Problem 3 — Gradient clipping
def clip_gradients_by_norm(grads, max_norm=1.0):
    total_norm = 0
    for g in grads.values():
        total_norm += np.sum(g ** 2)
    total_norm = np.sqrt(total_norm)

    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-15)
        for k in grads:
            grads[k] *= scale
    return grads

def clip_gradients_by_value(grads, min_val=-1.0, max_val=1.0):
    for k in grads:
        grads[k] = np.clip(grads[k], min_val, max_val)
    return grads

# Verify clipping works
test_grads = {'dW1': np.array([[100.0, -200.0], [50.0, 300.0]]),
              'db1': np.array([[500.0]])}

clipped_norm = clip_gradients_by_norm(test_grads, max_norm=1.0)
print("Gradient norm after clipping:",
      np.sqrt(sum(np.sum(g**2) for g in clipped_norm.values())))

clipped_val = clip_gradients_by_value(test_grads, -0.5, 0.5)
print(f"Clipped by value:\ndW1:\n{clipped_val['dW1']}")

## Summary

- **Backpropagation** applies the chain rule to compute gradients efficiently through the network
- **Computational graphs** represent forward (left→right) and backward (right→left) passes
- **Gradient descent variants**: Batch (full dataset), Mini-batch (batches of 32-256), SGD (single sample)
- **Vanishing gradients** occur when gradients shrink → use He init + ReLU
- **Exploding gradients** occur when gradients grow → use gradient clipping
- **Weight initialization** is critical: Xavier for tanh/sigmoid, He for ReLU
- **Learning rate schedules** (step, exponential, cosine) improve convergence